# DCT1401 - Inteligência Artificial (UFRN)
## Atividade Prática Avaliativa: Limpeza e Pré-processamento dos Dados
**Aluno:** Wallison Valdemiro Silvino Dias  
**Matrícula:** 20250023771  
**Dataset:** Greek Urban Air Quality & Health Impact Dataset (2020-2024)

---
### Objetivos Desta Etapa
1. **Tratamento e Verificação de Nulos:** Validação de integridade e definição de diretrizes defensivas.
2. **Tratamento de Outliers:** Justificativa da abordagem (Winsorização / preservação fundamentada) e análise de impacto.
3. **Codificação de Variáveis Categóricas:** Aplicação de One-Hot Encoding (OHE) em variáveis nominais (`city`) e Label/Ordinal Encoding para o target sanitário.
4. **Normalização/Padronização:** Avaliação e aplicação de `StandardScaler` (Z-score) para algoritmos sensíveis a distâncias e gradientes.
5. **Geração dos Datasets Finais:** Exportação de `dataset_clean.csv`, `dataset_regression.csv` e `dataset_classification.csv`.

In [1]:
import sys
from pathlib import Path

root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import numpy as np
import pandas as pd
from src import data_loader, preprocessing, visualization

print("Módulos carregados com sucesso.")

Módulos carregados com sucesso.


### 1. Limpeza Inicial e Engenharia de Atributos Temporais
Convertemos o atributo `timestamp` para o formato datetime nativo e derivamos features temporais cíclicas relevantes: `hour`, `month`, `day_of_week` e `is_weekend`.

In [2]:
raw_df = data_loader.load_raw_data()
clean_df = preprocessing.clean_dataset(raw_df)

print(f"Dataset limpo: {clean_df.shape[0]:,} linhas x {clean_df.shape[1]} colunas")
display(clean_df[["timestamp", "hour", "month", "day_of_week", "is_weekend"]].head())

Dataset limpo: 453,096 linhas x 36 colunas


,timestamp,hour,month,day_of_week,is_weekend
0,2020-01-01,0,1,2,0
1,2020-01-01,0,1,2,0
2,2020-01-01,0,1,2,0
3,2020-01-01,0,1,2,0
4,2020-01-01,0,1,2,0


### 2. Justificativa do Tratamento de Outliers
No domínio de monitoramento ambiental, descartar outliers ingênuamente excluiria os cenários mais graves de contaminação e risco populacional (queimadas e poeira do Saara).
- Para **modelos baseados em árvores (Random Forest, XGBoost)**, as observações extremas são mantidas na íntegra no dataset limpo, visto que esses algoritmos realizam partições ortogonais invariantes a escalas monotônicas.
- Para **modelos lineares e baseados em distância (KNN, SVM, Regressão Linear)**, os atributos numéricos contínuos recebem Winsorização defensiva ou padronização Z-score robusta, mitigando alavancagem excessiva.

In [3]:
# Exemplo de verificação de quantis para variáveis extremas
extreme_cols = ["PM2_5_ugm3", "PM10_ugm3", "AQI", "Traffic_Density_Index"]
quantiles_df = clean_df[extreme_cols].quantile([0.01, 0.25, 0.50, 0.75, 0.99, 1.00])
display(quantiles_df.round(2))

,PM2_5_ugm3,PM10_ugm3,AQI,Traffic_Density_Index
0.01,1.00,1.00,20.3,10.2
0.25,8.21,14.27,56.4,26.9
0.50,14.75,26.69,86.3,40.7
0.75,23.96,43.65,129.7,63.8
0.99,58.62,109.42,290.6,158.6
1.00,166.11,336.10,729.1,194.3


### 3. Codificação de Variáveis Categóricas
- **One-Hot Encoding (OHE):** Aplicado às 12 cidades gregas (`city`). Como não há ordem hierárquica entre Atenas, Tessalônica e Creta, OHE evita que o modelo infira uma relação de ordem artificial. Utiliza-se `drop_first=True` para evitar colinearidade estrita (*dummy variable trap*).
- **Label / Ordinal Encoding:** Aplicado à variável alvo `Outdoor_Activity_Recommendation` para classificação:
  - `0`: All activities safe
  - `1`: Sensitive groups limit outdoor activity
  - `2`: Reduce prolonged outdoor activity
  - `3`: Avoid outdoor activity

In [4]:
# Demonstração de codificação
city_dummies = pd.get_dummies(clean_df["city"], prefix="city", drop_first=True, dtype=int)
print(f"Número de colunas dummies geradas para cidades: {city_dummies.shape[1]}")
display(city_dummies.head(3))

Número de colunas dummies geradas para cidades: 11


,city_Athens,city_Chania,city_Heraklion,city_Ioannina,city_Kavala,city_Kozani,city_Larissa,city_Patras,city_Rhodes,city_Thessaloniki,city_Volos
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,0


### 4. Normalização / Padronização
Variáveis numéricas em diferentes ordens de grandeza (ex: `population` na casa dos milhões vs `Wind_Speed_ms` de 0 a 9) causam distorção severa em algoritmos que calculam distâncias euclidianas (KNN, SVM) ou otimizam via gradiente descendente.
Utilizamos o **`StandardScaler`** ($z = \frac{x - \mu}{\sigma}$), garantindo média 0 e variância unitária.

### 5. Geração e Salvamento dos Datasets Processados
Executamos o pipeline completo para gerar os três datasets exigidos na pasta `data/processed/`:
1. `dataset_clean.csv`: Dataset limpo com features temporais derivadas.
2. `dataset_regression.csv`: Features padronizadas, dummies de cidade e target contínuo `AQI`.
3. `dataset_classification.csv`: Medições de poluentes, features climáticas padronizadas e target discreto `target_class` (0 a 3).

In [5]:
# Executa a geração e salvamento
print("Iniciando geração e salvamento dos datasets processados...")
clean_path, reg_path, clf_path = preprocessing.save_all_processed_datasets(raw_df)

print(f"1. Dataset Limpo: {clean_path} (Tamanho: {clean_path.stat().st_size / (1024*1024):.2f} MB)")
print(f"2. Dataset Regressão: {reg_path} (Tamanho: {reg_path.stat().st_size / (1024*1024):.2f} MB)")
print(f"3. Dataset Classificação: {clf_path} (Tamanho: {clf_path.stat().st_size / (1024*1024):.2f} MB)")

Iniciando geração e salvamento dos datasets processados...


1. Dataset Limpo: C:\Users\walli\OneDrive\Documentos\bsi_ofc\4-semestre\ia\air-quality-data\data\processed\dataset_clean.csv (Tamanho: 105.73 MB)
2. Dataset Regressão: C:\Users\walli\OneDrive\Documentos\bsi_ofc\4-semestre\ia\air-quality-data\data\processed\dataset_regression.csv (Tamanho: 111.31 MB)
3. Dataset Classificação: C:\Users\walli\OneDrive\Documentos\bsi_ofc\4-semestre\ia\air-quality-data\data\processed\dataset_classification.csv (Tamanho: 106.12 MB)


In [6]:
# Verificação das primeiras linhas do dataset de regressão
reg_df = data_loader.load_regression_data()
print("Dataset de Regressão:")
display(reg_df.head(3))

Dataset de Regressão:


,latitude,longitude,population,industrial_index,Temperature_C,Humidity_pct,Wind_Speed_ms,Wind_Direction_deg,Pressure_hPa,Rainfall_mm,...,city_Heraklion,city_Ioannina,city_Kavala,city_Kozani,city_Larissa,city_Patras,city_Rhodes,city_Thessaloniki,city_Volos,AQI
0,40.855414,25.860298,-0.622304,-0.944594,-0.821166,-0.007568,1.308819,-0.882323,-0.603230,0.675776,...,0,0,0,0,0,0,0,0,0,80.8
1,40.896534,25.833779,-0.622304,-0.944594,-1.462579,0.317619,-1.728630,1.263512,1.097719,-0.427117,...,0,0,0,0,0,0,0,0,0,56.4
2,37.987214,23.701981,2.178534,-0.057248,-1.154207,1.071462,0.005913,-0.584023,0.397328,-0.427117,...,0,0,0,0,0,0,0,0,0,78.0


In [7]:
# Verificação das primeiras linhas do dataset de classificação
clf_df = data_loader.load_classification_data()
print("Dataset de Classificação:")
display(clf_df.head(3))

Dataset de Classificação:


,PM2_5_ugm3,PM10_ugm3,NO2_ugm3,O3_ugm3,SO2_ugm3,CO_mgm3,Temperature_C,Humidity_pct,Wind_Speed_ms,UV_Index,...,city_Heraklion,city_Ioannina,city_Kavala,city_Kozani,city_Larissa,city_Patras,city_Rhodes,city_Thessaloniki,city_Volos,target_class
0,-0.331922,-0.384969,-0.067573,-0.186398,-1.539756,-0.206243,-0.821166,-0.007568,1.308819,-0.746896,...,0,0,0,0,0,0,0,0,0,1
1,-0.684449,-0.674860,-0.581224,-0.283756,-0.330170,-1.256966,-1.462579,0.317619,-1.728630,-0.746896,...,0,0,0,0,0,0,0,0,0,1
2,-0.447631,-0.132350,0.246006,-0.276002,-0.381433,-0.569955,-1.154207,1.071462,0.005913,-0.746896,...,0,0,0,0,0,0,0,0,0,1
